In [ ]:
# Complete, hardened EWC script — copy & run on Kaggle
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import random
import warnings
import time

warnings.filterwarnings("ignore")

# -----------------------
#  Model
# -----------------------
class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(nn.Conv2d(in_c, out_c, 1, stride, bias=False),
                                          nn.BatchNorm2d(out_c))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        return F.relu(out)

class Net(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3,64,3,1,1,bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = ResidualBlock(64,64)
        self.layer2 = ResidualBlock(64,128,2)
        self.layer3 = ResidualBlock(128,256,2)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)
    def forward(self,x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x).view(x.size(0), -1)
        return self.fc(x)

# -----------------------
#  Utilities
# -----------------------
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def evaluate(model, loader, device):
    model.eval()
    correct = 0; total = 0
    with torch.no_grad():
        for x,y in loader:
            x,y = x.to(device), y.to(device)
            out = model(x)
            pred = out.argmax(1)
            correct += (pred==y).sum().item()
            total += y.size(0)
    return 100.0 * correct / total

# -----------------------
#  Fisher (robust)
# -----------------------
def compute_fisher(model, loader, device, max_samples=512):
    """
    Empirical Fisher approx.
    - iterates batches; for each example computes loss on its true label and accumulates grad^2
    - returns dict(name -> tensor) on same device as model
    """
    model.eval()
    fisher = {n: torch.zeros_like(p, device=device) for n,p in model.named_parameters() if p.requires_grad}
    seen = 0
    for x_batch, y_batch in loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        batch_size = x_batch.size(0)
        # do per-sample: split batch to avoid memory explosion
        for i in range(batch_size):
            model.zero_grad()
            xi = x_batch[i:i+1]
            yi = y_batch[i:i+1]
            out = model(xi)
            loss = F.cross_entropy(out, yi)
            loss.backward()
            for n,p in model.named_parameters():
                if p.requires_grad and p.grad is not None:
                    fisher[n] += (p.grad.detach() ** 2)
            seen += 1
            if seen >= max_samples:
                break
        if seen >= max_samples:
            break
    # normalize
    if seen == 0:
        raise RuntimeError("No samples found when computing Fisher — loader empty?")
    for n in fisher:
        fisher[n] /= float(seen)
        # numeric safety: tiny floor
        fisher[n] = fisher[n].clamp(min=1e-8)
    return fisher

# -----------------------
#  Main EWC training function
# -----------------------
def run_ewc_seed(seed, loaders, device, lambda_ewc=50.0, fisher_samples=512):
    """
    loaders = (trainA_loader, trainB_loader, testA_loader, testB_loader)
    """
    set_seed(seed)
    trainA, trainB, testA, testB = loaders

    model = Net(num_classes=10).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    epochs_A = 10
    epochs_B = 40

    # Phase A
    for ep in range(epochs_A):
        model.train()
        for x,y in trainA:
            x,y = x.to(device), y.to(device)
            opt.zero_grad()
            out = model(x)
            loss = F.cross_entropy(out, y)
            loss.backward()
            opt.step()

    acc_A_init = evaluate(model, testA, device)

    # Compute Fisher on trainA (safe)
    fisher = compute_fisher(model, trainA, device, max_samples=fisher_samples)
    # save old params
    old_params = {n: p.clone().detach() for n,p in model.named_parameters() if p.requires_grad}

    # Phase B with EWC penalty
    for ep in range(epochs_B):
        model.train()
        for x,y in trainB:
            x,y = x.to(device), y.to(device)
            opt.zero_grad()
            out = model(x)
            loss_task = F.cross_entropy(out, y)

            # EWC penalty (vectorized)
            ewc_penalty = 0.0
            for n, p in model.named_parameters():
                if p.requires_grad:
                    # using fisher[n] already on device
                    ewc_penalty = ewc_penalty + (fisher[n] * (p - old_params[n])**2).sum()
            loss_total = loss_task + (lambda_ewc * 0.5) * ewc_penalty

            # safeguard: if loss_total is NaN or huge, clamp
            if torch.isnan(loss_total) or torch.isinf(loss_total):
                # print debug and skip update to avoid destroying weights
                print("[DEBUG] loss_total invalid; skipping step")
                continue

            loss_total.backward()
            opt.step()

    acc_A_final = evaluate(model, testA, device)
    acc_B_final = evaluate(model, testB, device)

    # small debug if catastrophic forgetting occurs
    if acc_A_final < 1.0:
        model.eval()
        # show predictions distribution on a small subset of testA
        preds = []
        with torch.no_grad():
            cnt = 0
            for x,y in testA:
                x = x.to(device)
                out = model(x)
                preds.append(out.argmax(1).cpu().numpy())
                cnt += x.size(0)
                if cnt >= 128:
                    break
        preds = np.concatenate(preds)[:128]
        (unique, counts) = np.unique(preds, return_counts=True)
        dist = dict(zip(unique.tolist(), counts.tolist()))
        print("[DEBUG] small testA predicted class counts (first 128 samples):", dist)
        # print classifier weight norms
        norms = {}
        for n,p in model.named_parameters():
            if 'fc.weight' in n or 'fc.bias' in n:
                norms[n] = p.norm().item()
        print("[DEBUG] classifier param norms (sample):", norms)

    return {"acc_A_init": acc_A_init, "acc_A_final": acc_A_final, "acc_B_final": acc_B_final}

# -----------------------
#  Prepare data loaders (CIFAR10 split 0-4 / 5-9)
# -----------------------
def make_loaders(batch_size=256, replay_size=None, num_workers=2):
    stats = ((0.4914,0.4822,0.4465),(0.247,0.243,0.261))
    t_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(*stats)
    ])
    t_test = transforms.Compose([transforms.ToTensor(), transforms.Normalize(*stats)])

    root = '/kaggle/input/cifar-10'
    train_ds = torchvision.datasets.CIFAR10(root=root, train=True, download=False, transform=t_train)
    test_ds  = torchvision.datasets.CIFAR10(root=root, train=False, download=False, transform=t_test)

    idxA = [i for i,l in enumerate(train_ds.targets) if l < 5]
    idxB = [i for i,l in enumerate(train_ds.targets) if l >= 5]
    tA   = [i for i,l in enumerate(test_ds.targets) if l < 5]
    tB   = [i for i,l in enumerate(test_ds.targets) if l >= 5]

    if replay_size is None:
        replay_size = len(idxA)

    loaders = (
        DataLoader(Subset(train_ds, idxA), batch_size, shuffle=True, num_workers=num_workers, pin_memory=True, persistent_workers=True),
        DataLoader(Subset(train_ds, idxB), batch_size, shuffle=True, num_workers=num_workers, pin_memory=True, persistent_workers=True),
        DataLoader(Subset(test_ds, tA), batch_size, shuffle=False, num_workers=0, pin_memory=False),
        DataLoader(Subset(test_ds, tB), batch_size, shuffle=False, num_workers=0, pin_memory=False)
    )
    return loaders

# -----------------------
#  Run multi-seed experiment
# -----------------------
def run_all(seeds=[0,1,2,3,4], device_str=None):
    device = torch.device(device_str if device_str is not None else ("cuda" if torch.cuda.is_available() else "cpu"))
    print("🚀 Device:", device)
    loaders = make_loaders(batch_size=256, num_workers=2)
    results = []
    start = time.time()
    for s in seeds:
        print(f"\n-> Running EWC seed {s} ...")
        res = run_ewc_seed(s, loaders, device, lambda_ewc=50.0, fisher_samples=512)
        print(f"   ↳ Seed {s} Results: Task A Init={res['acc_A_init']:.2f}% | Task A Final={res['acc_A_final']:.2f}% | Task B Final={res['acc_B_final']:.2f}%")
        results.append(res)
    # aggregate
    A_init = np.array([r["acc_A_init"] for r in results])
    A_final = np.array([r["acc_A_final"] for r in results])
    B_final = np.array([r["acc_B_final"] for r in results])
    ret = (A_final / (A_init + 1e-12)) * 100.0
    forg = A_init - A_final
    bwt = A_final - A_init
    print("\n=== MEAN ± STD ===")
    print(f"Task A Init : {A_init.mean():.2f} ± {A_init.std():.2f}")
    print(f"Task A Final: {A_final.mean():.2f} ± {A_final.std():.2f}")
    print(f"Task B Final: {B_final.mean():.2f} ± {B_final.std():.2f}")
    print(f"Retention   : {ret.mean():.2f}% ± {ret.std():.2f}")
    print(f"Forgetting  : {forg.mean():.2f} ± {forg.std():.2f}")
    print(f"BWT         : {bwt.mean():.2f} ± {bwt.std():.2f}")
    print("Total wall time (s):", time.time() - start)

if __name__ == "__main__":
    run_all()

🚀 Device: cuda

-> Running EWC seed 0 ...
[DEBUG] small testA predicted class counts (first 128 samples): {5: 22, 6: 25, 7: 8, 8: 30, 9: 43}
[DEBUG] classifier param norms (sample): {'fc.weight': 5.906852722167969, 'fc.bias': 0.23524202406406403}
   ↳ Seed 0 Results: Task A Init=78.04% | Task A Final=0.00% | Task B Final=83.86%

-> Running EWC seed 1 ...
[DEBUG] small testA predicted class counts (first 128 samples): {5: 24, 6: 30, 7: 12, 8: 29, 9: 33}
[DEBUG] classifier param norms (sample): {'fc.weight': 5.702686309814453, 'fc.bias': 0.25254666805267334}
   ↳ Seed 1 Results: Task A Init=81.94% | Task A Final=0.00% | Task B Final=88.22%

-> Running EWC seed 2 ...
[DEBUG] small testA predicted class counts (first 128 samples): {5: 23, 6: 32, 7: 20, 8: 22, 9: 31}
[DEBUG] classifier param norms (sample): {'fc.weight': 5.6002912521362305, 'fc.bias': 0.2140769213438034}
   ↳ Seed 2 Results: Task A Init=85.92% | Task A Final=0.00% | Task B Final=89.92%

-> Running EWC seed 3 ...
